# Visualise the neighbour-exchange channel

Self-contained. Set the base potential and one gate geometry below, then run all.
The gate parameter reported by `search.py` is `n_delta = 1 - n_target`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from vbc_landscape import swap_landscape, find_swap_endpoints, minimax_barrier

# base potential
R_REP, R_ATT, E_R, E_A = 1.0, 1.5, 20.0, -2.0

# one gate geometry to look at  (r_low, r_high, n_target);  n_delta = 1 - n_target
R_LOW, R_HIGH, N_TARGET = 1.29, 1.50, 0.70

VALENCIES = [2, 4, 6]
r_grid = np.linspace(0.9, 2.2, 401)

In [ ]:
def draw_channel(ax, valency):
    U = swap_landscape(valency, r_grid, R_LOW, R_HIGH, N_TARGET,
                       r_rep=R_REP, r_att=R_ATT, E_r=E_R, E_a=E_A)
    ext = [r_grid[0], r_grid[-1], r_grid[0], r_grid[-1]]
    im = ax.imshow((U / abs(E_A)).T, origin="lower", extent=ext, aspect="equal",
                   vmin=-valency, vmax=0.0, cmap="viridis")
    ax.set_xlabel(r"$r_A$")
    ax.set_ylabel(r"$r_B$")
    title = f"f = {valency}"
    ep = find_swap_endpoints(U, r_grid)
    if ep is not None:
        B = minimax_barrier(U, ep[0], ep[1])
        B_rel = B - max(U[ep[0]], U[ep[1]])
        ax.contour(r_grid, r_grid, U.T, levels=[B], colors="r", linewidths=1.0)
        for (i, j) in ep:
            ax.plot(r_grid[i], r_grid[j], "r*", ms=13)
        title += f"    B/|E_a| = {B_rel / abs(E_A):.3f}"
    ax.set_title(title)
    return im

fig, axes = plt.subplots(1, len(VALENCIES), figsize=(4.2 * len(VALENCIES), 4.0))
axes = np.atleast_1d(axes)
im = None
for ax, v in zip(axes, VALENCIES):
    im = draw_channel(ax, v)
fig.colorbar(im, ax=list(axes), label=r"$U / |E_a|$", shrink=0.85)
plt.show()

## Read back a search result

Point at a `top*_barrier.csv` written by `search.py`.

In [ ]:
import csv, os

path = "./outputs/top100_barrier.csv"
if os.path.exists(path):
    with open(path) as f:
        best = next(csv.DictReader(f))
    print("best gate   r_low=%s  r_high=%s  n_target=%s  (n_delta=%s)"
          % (best["r_low"], best["r_high"], best["n_target"], best["n_delta"]))
    print("worst-case  B/|E_a| = %s    spur/|E_a| = %s"
          % (best["bEa_max"], best["spur_max"]))
    fs = sorted(int(k[5:]) for k in best if k.startswith("bEa_v"))
    bvals = [float(best[f"bEa_v{f}"]) for f in fs]
    fig, ax = plt.subplots(figsize=(4.0, 3.0))
    ax.plot(fs, bvals, "o-")
    ax.set_xlabel("valency f")
    ax.set_ylabel(r"$B / |E_a|$")
    ax.set_title("exchange barrier of the best gate")
    plt.show()
else:
    print("Run search.py first, e.g.  python search.py --out_dir ./outputs")

## Spurious (off-channel) minima

The profile is the total energy of a particle approaching an already-saturated
neighbour, including the back-reaction on the neighbour's existing bonds. It is drawn
relative to the N-far baseline, so a dip below zero is a transient trap and its depth
is the value returned by `sporadic_min_depth`.

In [ ]:
from vbc_landscape import density_phi, omega_gate, V_rep, V_att, sporadic_min_depth

def spurious_profile(valency, r_low, r_high, n_target, *, r_rep, r_att, E_r, E_a, n_r=501):
    r = np.linspace(r_rep, r_att, n_r)
    ncut = valency + (n_target - 1.0)
    r_star = 0.5 * (r_rep + r_att)
    rho_rs = float(density_phi(np.array([r_star]), r_low, r_high)[0])
    vatt_rs = float(V_att(np.array([r_star]), r_rep, r_att, E_a)[0])
    W_S = float(omega_gate(np.array([rho_rs]), ncut)[0])
    rho_r = density_phi(r, r_low, r_high)
    W_C = omega_gate(valency * rho_rs + rho_r, ncut)
    W_N = omega_gate(rho_r, ncut)
    U = (V_rep(r, r_rep, E_r) + W_C * W_N * V_att(r, r_rep, r_att, E_a)
         + valency * W_C * W_S * vatt_rs)
    baseline = valency * float(omega_gate(np.array([valency * rho_rs]), ncut)[0]) * W_S * vatt_rs
    return r, U, baseline

def draw_spurious(ax, name, gate, valencies):
    r_low, r_high, n_target = gate
    for f in valencies:
        r, U, base = spurious_profile(f, r_low, r_high, n_target,
                                      r_rep=R_REP, r_att=R_ATT, E_r=E_R, E_a=E_A)
        d = sporadic_min_depth(f, r_low, r_high, n_target,
                               r_rep=R_REP, r_att=R_ATT, E_r=E_R, E_a=E_A)
        ax.plot(r, (U - base) / abs(E_A), label=f"f={f}   depth={d:.3f}")
    ax.axhline(0.0, color="k", lw=0.8, ls="--")
    ax.set_xlabel(r"$r_{CN}$")
    ax.set_ylabel(r"$(U - U_\infty) / |E_a|$")
    ax.set_title(name)
    ax.legend(fontsize=7)

In [ ]:
# the three gates from the RCS paper, on the shared base potential above
# VBC(r_low, r_high, n_delta);  n_target = 1 - n_delta
SETUPS = [
    ("standard  VBC(1.5, 2.0, 0)",         (1.5,   2.0,   1.00)),
    ("looser  VBC(1.224, 1.561, -0.03)",   (1.224, 1.561, 1.03)),
    ("optimised  VBC(1.29, 1.50, 0.30)",   (1.29,  1.50,  0.70)),
]
VALS = [2, 3, 4, 5, 6]

fig, axes = plt.subplots(1, 3, figsize=(13.5, 4.0), sharey=True)
for ax, (name, gate) in zip(axes, SETUPS):
    draw_spurious(ax, name, gate, VALS)
plt.tight_layout()
plt.show()

## Random passive placement

With `key="random"` the passive neighbours are scattered uniformly in the binding
zone rather than sitting at the well centre, so each contributes its own density
weight to the count. The depth is averaged over `n_sample` placements. Note that a
random placement need not keep the centre saturated, so the mean blends genuine
binding with true spurious traps.

In [ ]:
from vbc_landscape import spurious_depth

N_SAMPLE = 200
fig, ax = plt.subplots(figsize=(5.5, 3.8))
for name, gate in SETUPS:
    rl, rh, nt = gate
    avg = [spurious_depth(f, rl, rh, nt, r_rep=R_REP, r_att=R_ATT, E_r=E_R, E_a=E_A,
                          key="random", n_sample=N_SAMPLE, seed=0) for f in VALS]
    ax.plot(VALS, avg, "o-", label=name)
    print(name, " ", "  ".join(f"f{f}={a:.3f}" for f, a in zip(VALS, avg)))
ax.set_xlabel("valency f")
ax.set_ylabel(r"mean spurious depth $/|E_a|$")
ax.set_title(f"random passive placement, mean over n_sample={N_SAMPLE}")
ax.legend(fontsize=7)
plt.show()